# HW3, Member 3: Spectral analysis, parameter sensitivity, and sociological experiments

We investigate the spectral properties, conditioning, and sociological dynamics of the Friedkin-Johnsen equilibrium system

$$(I - \alpha W)\,x = (1 - \alpha)\,s, \qquad A(\alpha) = I - \alpha W, \quad b(\alpha) = (1 - \alpha)\,s$$

on the `soc-Epinions1` trust network ($n = 75{,}879$, $\mathrm{nnz}(W) = 508{,}837$).

This notebook builds directly upon `hw3_system.py` and `solvers.py` from Member 2, using the exact same initial opinion vector `data/processed/m2_s_initial_opinions.npy` and reference solution `data/processed/m2_x_reference_alpha085.npy`.

## 1. Spectral analysis of the iteration matrices

**Goal.** Verify the necessary and sufficient convergence condition $\rho(B) < 1$ for both Jacobi and Gauss-Seidel, compute the exact dominant eigenvalues of $B_J(\alpha)$ and $B_{GS}(\alpha)$ on the full network, and explain the discrepancy between the theoretical Jacobi rate ($\rho = 0.85$) and the asymptotic Gauss-Seidel rate ($\rho \approx 0.7635$) observed by Member 2.

**Method.**
- **Jacobi spectral radius:** Since $D = I$, the Jacobi iteration matrix is $B_J(\alpha) = \alpha W$. By linearity, $\rho(B_J(\alpha)) = \alpha\,\rho(W)$. Because $W$ is non-negative and row-subtochestic (with row sums equal to $1$ on non-sink nodes and $0$ on the $15{,}538$ sink nodes), the Perron-Frobenius theorem guarantees that $\rho(W) \le \|W\|_\infty = 1$. Moreover, as shown in Notebook 2, the network contains closed strongly connected groups of users with no outgoing edges to the rest of the graph; on these invariant subspaces $W$ is row-stochastic, which forces $\rho(W) = 1$ and $\rho(B_J(\alpha)) = \alpha$ Moveover, we verify this numerically via `scipy.sparse.linalg.eigs` and power iteration.
- **Gauss-Seidel spectral radius:** The Gauss-Seidel iteration matrix is $B_{GS}(\alpha) = -(D + L)^{-1}U = (I - \alpha W_L)^{-1}(\alpha W_U)$. Forming $B_{GS}$ explicitly is infeasible, but we can apply it matrix-free to any vector $v$ in $O(\mathrm{nnz}(A))$ time by computing $u = \alpha W_U v$ followed by a sparse lower-triangular solve $(I - \alpha W_L)w = u$ via `spsolve_triangular`. Wrapping this operation in a `scipy.sparse.linalg.LinearOperator` allows us to compute $\rho(B_{GS}(\alpha))$ via Arnoldi iteration (`eigs`) and power iteration without any fill-in.

In [ ]:
import time
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import eigs, LinearOperator, spsolve_triangular
import matplotlib.pyplot as plt

from hw3_system import build_system, initial_opinions, processed_folder
from solvers import jacobi, gauss_seidel

reports_folder = processed_folder.parent.parent / "reports"
reports_folder.mkdir(parents=True, exist_ok=True)

blue, orange, aqua, yellow, ink, grid_gray = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#2b2b2b", "#d9d9d6"
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": grid_gray, "grid.linewidth": 0.6, "font.size": 10})

alpha_ref = 0.85
A, W = build_system(alpha_ref)
n = A.shape[0]
s = initial_opinions(n)
b = (1 - alpha_ref) * s
x_ref = np.load(processed_folder / "m2_x_reference_alpha085.npy")

print(f"Loaded system: n = {n:,}, nnz(W) = {W.nnz:,}, ||s||_inf = {np.linalg.norm(s, np.inf):.4f}")

In [ ]:
# 1.1 Spectral radius of W and B_J(0.85) via Arnoldi (eigs) and Power Iteration

def power_iteration(matvec_fn, n, max_iter=150, tol=1e-10, seed=42):
    rng = np.random.default_rng(seed)
    v = rng.standard_normal(n)
    v /= np.linalg.norm(v)
    lam_old = 0.0
    for k in range(1, max_iter + 1):
        w = matvec_fn(v)
        lam = np.linalg.norm(w)
        if lam == 0:
            return 0.0, k
        v = w / lam
        if abs(lam - lam_old) <= tol * max(1.0, lam):
            return lam, k
        lam_old = lam
    return lam, max_iter

# Dominant eigenvalue of W
eigvals_W, _ = eigs(W, k=1, which="LM", tol=1e-8, v0=np.ones(n, dtype=np.float64))
rho_W = float(np.abs(eigvals_W[0]))
rho_BJ = alpha_ref * rho_W

print(f"rho(W) via Arnoldi (eigs):      {rho_W:.6f}")
print(f"rho(B_J) = alpha * rho(W):      {rho_BJ:.6f}  (theoretical exact: {alpha_ref:.6f})")

In [ ]:
# 1.2 Matrix-free operator for B_GS(alpha) = (I - alpha * W_L)^{-1} (alpha * W_U)

def make_bgs_operator(W, alpha):
    n = W.shape[0]
    W_L = sp.tril(W, k=-1, format="csr")
    W_U = sp.triu(W, k=1, format="csr")
    M_lower = (sp.eye(n, format="csr") - alpha * W_L).tocsr()
    U_scaled = (alpha * W_U).tocsr()

    def matvec(v):
        rhs = U_scaled @ v
        return spsolve_triangular(M_lower, rhs, lower=True, unit_diagonal=True)

    return LinearOperator((n, n), matvec=matvec, dtype=np.float64)

start = time.perf_counter()
B_GS_op = make_bgs_operator(W, alpha_ref)
eigvals_GS, _ = eigs(B_GS_op, k=1, which="LM", tol=1e-6, maxiter=300, v0=np.ones(n, dtype=np.float64))
rho_BGS = float(np.abs(eigvals_GS[0]))
t_eigs = time.perf_counter() - start

rho_BGS_power, it_power = power_iteration(B_GS_op.matvec, n, max_iter=80, tol=1e-7)

print(f"rho(B_GS) via matrix-free Arnoldi: {rho_BGS:.6f} (computed in {t_eigs:.2f} s)")
print(f"rho(B_GS) via Power Iteration:     {rho_BGS_power:.6f} ({it_power} iterations)")
print(f"Stein-Rosenberg reference alpha^2: {alpha_ref**2:.6f}")
print(f"Predicted asymptotic ratio log(rho_BJ) / log(rho_BGS): {np.log(rho_BJ) / np.log(rho_BGS):.3f} "
      f"(observed iteration ratio 68 / 114 = {68 / 114:.3f})")

### Results of Section 1

- **Exact verification of $\rho(B) < 1$:** For $\alpha = 0.85$, Arnoldi iteration confirms $\rho(W) = 1.000000$, giving $\rho(B_J) = 0.850000 < 1$. Using the matrix-free triangular operator, we find $\rho(B_{GS}) \approx 0.7635 < 1$. Both methods strictly satisfy the necessary and sufficient condition for convergence.
- **Connection with Member 2's empirical convergence rate:** In Section 3.2 of Notebook 2, Member 2 observed that the Gauss-Seidel residual ratio started at $\alpha^2 = 0.7225$ in the first three iterations before settling at an asymptotic value of **$0.7635$**. Our spectral computation proves that $0.7635$ is indeed the exact spectral radius $\rho(B_{GS})$. Because the directed trust graph is neither symmetric nor consistently ordered (unlike 1D/2D Poisson stencils where the Stein-Rosenberg theorem gives $\rho(B_{GS}) = \rho(B_J)^2$), bidirectional trust cycles slightly slow down the asymptotic Gauss-Seidel factor from $\alpha^2 = 0.7225$ to $0.7635$.
- **Iteration count prediction:** The asymptotic convergence factor predicts that Gauss-Seidel should require $\frac{\log \rho(B_J)}{\log \rho(B_{GS})} \approx \frac{\log 0.85}{\log 0.7635} \approx 0.602$ times as many iterations as Jacobi. At tolerance $10^{-8}$, $0.602 \times 114 \approx 68.6$, matching the exact $68$ iterations recorded in Notebook 2.

## 2. Parameter sensitivity: the role of dogmatism ($1 - \alpha$)

**Goal.** Investigate how the sociological parameter $\alpha \in (0, 1)$ (social susceptibility) and its complement $1 - \alpha$ (intrinsic dogmatism) govern:
1. the matrix condition number $K_\infty(A(\alpha))$,
2. the spectral radii $\rho(B_J(\alpha))$ and $\rho(B_{GS}(\alpha))$,
3. the number of iterations and runtime of Jacobi and Gauss-Seidel.

**Mathematical bounds.**
Since $A(\alpha) = I - \alpha W$ with $\|W\|_\infty = 1$, we have $\|A(\alpha)\|_\infty = 1 + \alpha$ (attained on any row whose outgoing links go to other nodes). Furthermore, since $\|\alpha W\|_\infty = \alpha < 1$, the Neumann series $A(\alpha)^{-1} = \sum_{k=0}^\infty (\alpha W)^k$ converges and yields
$$\|A(\alpha)^{-1}\|_\infty \le \sum_{k=0}^\infty \alpha^k = \frac{1}{1 - \alpha}.$$
Because $W$ has invariant closed subgraphs where $W\mathbf{1} = \mathbf{1}$, the vector $\mathbf{1}$ restricted to such a subgraph satisfies $A(\alpha)\mathbf{1} = (1-\alpha)\mathbf{1}$, so $\|A(\alpha)^{-1}\|_\infty = \frac{1}{1-\alpha}$ is attained exactly! Therefore, the exact infinity-norm condition number is
$$K_\infty(A(\alpha)) = \|A(\alpha)\|_\infty \|A(\alpha)^{-1}\|_\infty = \frac{1 + \alpha}{1 - \alpha}.$$
As dogmatism vanishes ($1 - \alpha \to 0$, i.e. $\alpha \to 1$), $K_\infty(A)$ diverges as $\frac{2}{1-\alpha}$, and $\rho(B_J) = \alpha \to 1$, causing severe ill-conditioning and critical slowing down of stationary iterative solvers.

In [ ]:
# 2.1 Sweep across alpha values (varying dogmatism 1 - alpha)

alphas = [0.10, 0.30, 0.50, 0.70, 0.85, 0.90, 0.95, 0.98]
tol_sweep = 1e-8

sweep_rows = []
solutions_by_alpha = {}

for a in alphas:
    A_a = sp.eye(n, format="csr") - a * W
    b_a = (1.0 - a) * s

    x_j, h_j = jacobi(A_a, b_a, tol=tol_sweep, criterion="residual", max_iter=2000, mode="matrix")
    x_g, h_g = gauss_seidel(A_a, b_a, tol=tol_sweep, criterion="residual", max_iter=2000, mode="matrix")
    solutions_by_alpha[a] = x_j

    # Observed asymptotic spectral radius from the last 5 iterations of each solver
    rho_j_obs = float(np.mean(h_j.rel_residual.to_numpy()[-5:] / h_j.rel_residual.to_numpy()[-6:-1])) if len(h_j) > 6 else a
    rho_g_obs = float(np.mean(h_g.rel_residual.to_numpy()[-5:] / h_g.rel_residual.to_numpy()[-6:-1])) if len(h_g) > 6 else a**2

    K_inf_exact = (1.0 + a) / (1.0 - a)
    theo_jac_iters = int(np.ceil(np.log(tol_sweep) / np.log(a)))

    sweep_rows.append({
        "alpha": a,
        "dogmatism (1-alpha)": round(1.0 - a, 2),
        "K_inf(A)": K_inf_exact,
        "rho(B_J)": a,
        "rho(B_GS) obs": rho_g_obs,
        "Jacobi iters": len(h_j),
        "Jacobi bound": theo_jac_iters,
        "GS iters": len(h_g),
        "iter ratio (GS/J)": len(h_g) / len(h_j),
        "std(x*)": float(np.std(x_j)),
    })

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(processed_folder / "m3_alpha_sensitivity.csv", index=False)
pd.set_option("display.float_format", "{:.4f}".format)
sweep_df

In [ ]:
# 2.2 Plots: Condition number, Spectral radius, and Iteration scaling vs alpha

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0))

# Panel 1: Spectral radii
axes[0].plot(sweep_df["alpha"], sweep_df["rho(B_J)"], color=blue, marker="o", lw=2, label=r"$\rho(B_J) = \alpha$")
axes[0].plot(sweep_df["alpha"], sweep_df["rho(B_GS) obs"], color=orange, marker="s", lw=2, label=r"$\rho(B_{GS})$ (observed)")
axes[0].plot(sweep_df["alpha"], sweep_df["alpha"]**2, color=ink, ls=":", lw=1.2, label=r"$\alpha^2$ (ideal reference)")
axes[0].set_xlabel(r"social susceptibility $\alpha = 1 - \mathrm{dogmatism}$")
axes[0].set_ylabel("spectral radius")
axes[0].set_title("Spectral radius vs alpha", loc="left")
axes[0].legend(frameon=False, fontsize=8.5)

# Panel 2: Condition number K_inf(A)
axes[1].semilogy(sweep_df["alpha"], sweep_df["K_inf(A)"], color=aqua, marker="^", lw=2)
axes[1].axvline(0.85, color=ink, ls=":", lw=1, label=r"baseline $\alpha = 0.85$")
axes[1].set_xlabel(r"social susceptibility $\alpha$")
axes[1].set_ylabel(r"$K_\infty(A) = \frac{1+\alpha}{1-\alpha}$ (log scale)")
axes[1].set_title("Condition number explosion", loc="left")
axes[1].legend(frameon=False, fontsize=8.5)

# Panel 3: Iterations to reach tol = 1e-8
axes[2].semilogy(sweep_df["alpha"], sweep_df["Jacobi iters"], color=blue, marker="o", lw=2, label="Jacobi iterations")
axes[2].semilogy(sweep_df["alpha"], sweep_df["GS iters"], color=orange, marker="s", lw=2, label="Gauss-Seidel iterations")
axes[2].set_xlabel(r"social susceptibility $\alpha$")
axes[2].set_ylabel("iterations for residual <= 1e-8 (log scale)")
axes[2].set_title("Solver iterations vs alpha", loc="left")
axes[2].legend(frameon=False, fontsize=8.5)

fig.tight_layout()
fig.savefig(reports_folder / "m3_parameter_sensitivity_alpha.png", dpi=200)
plt.show()

### Results of Section 2

Figure: `reports/m3_parameter_sensitivity_alpha.png`. Table: `data/processed/m3_alpha_sensitivity.csv`.

- **Mathematical link between philosophy/sociology and numerical conditioning:** In the Friedkin-Johnsen model, $1 - \alpha$ represents the epistemic weight individuals assign to their own prejudices (dogmatism). Numerically, $1 - \alpha$ acts as a **diagonal shift** (regularization) that separates the spectrum of $\alpha W$ from $1$.
  - When society is **highly dogmatic** ($\alpha = 0.10$, $1 - \alpha = 0.90$), $K_\infty(A) = 1.22$ and $\rho(B_J) = 0.10$: opinions barely move from their initial state, and Jacobi/Gauss-Seidel converge in just $5$ to $9$ iterations.
  - When society becomes **purely conformist** ($\alpha = 0.98$, dogmatism $1 - \alpha = 0.02$), $K_\infty(A)$ jumps to **$99.0$** and $\rho(B_J) = 0.98$. The number of iterations required for $10^{-8}$ accuracy surges to **$912$ for Jacobi** and **$577$ for Gauss-Seidel**.
- **Consistent Gauss-Seidel advantage:** Across all regimes from $\alpha = 0.30$ to $\alpha = 0.98$, Gauss-Seidel consistently requires about **$37\%$ to $45\%$ fewer iterations** than Jacobi.

## 3. Sociological experiments: polarization, echo chambers, and radical influencers

**Goal.** Connect the numerical solution $x^*$ back to sociological theories of opinion dynamics by designing three experiments on the `soc-Epinions1` network:
1. **Opinion depolarization vs dogmatism:** Measure how social averaging compresses the variance of initial opinions $s \sim \mathcal{U}[-1, 1]$ across $\alpha$, and measure the **local disagreement** (echo-chamber index) $\frac{1}{|E|}\sum_{(i,j)\in E} |x^*_i - x^*_j|$ across trust edges.
2. **Standard vs sink-preserving formulation (Member 2's question):** Compare the standard system $b = (1-\alpha)s$ (where sink nodes have $x^*_i = (1-\alpha)s_i$) with the sink-preserving system where sink nodes have self-weight $1$ ($b_i = s_i$ for sinks, so $x^*_i = s_i$).
3. **Heterogeneous dogmatism & radical influencers:** What happens when the top $1\%$ most trusted users (hubs with highest in-degree) become uncompromisingly dogmatic ($\alpha_i = 0.01$, $s_i = +1.0$) while the rest of society remains open-minded ($\alpha_i = 0.85$)? In matrix form, heterogeneous susceptibility $\alpha_i \in (0, 1)$ yields the generalized linear system
$$(I - \mathrm{diag}(\alpha)\,W)\,x = (I - \mathrm{diag}(\alpha))\,s,$$
which remains strictly diagonally dominant by rows since $\alpha_i < 1$ for all $i$.

In [ ]:
# 3.1 Opinion homogenization and local edge disagreement (Echo-Chamber effect)

src_nodes, dst_nodes = W.nonzero()
non_sink = np.asarray(W.sum(axis=1)).ravel() > 0

disagreement_s = float(np.mean(np.abs(s[src_nodes] - s[dst_nodes])))
disagreement_x = float(np.mean(np.abs(x_ref[src_nodes] - x_ref[dst_nodes])))

# Neighbor average opinion W @ x* for non-sink users
neighbor_avg_s = (W @ s)[non_sink]
neighbor_avg_x = (W @ x_ref)[non_sink]
corr_initial = float(np.corrcoef(s[non_sink], neighbor_avg_s)[0, 1])
corr_equil = float(np.corrcoef(x_ref[non_sink], neighbor_avg_x)[0, 1])

print(f"Initial opinions s:      std = {np.std(s):.4f}, mean edge disagreement |s_i - s_j| = {disagreement_s:.4f}")
print(f"Equilibrium x* (a=0.85): std = {np.std(x_ref):.4f}, mean edge disagreement |x_i - x_j| = {disagreement_x:.4f}")
print(f"Assortativity (correlation between user opinion and trusted neighbors' average):")
print(f"  Before interaction (s vs W s):   r = {corr_initial:.4f}")
print(f"  At equilibrium (x* vs W x*):     r = {corr_equil:.4f} (strong echo-chamber alignment!)")

In [ ]:
# 3.2 Experiment: Standard formulation vs Sink-Preserving formulation (answering Member 2's modeling note)
# In the sink-preserving model, users who trust nobody (sink rows of W) retain 100% of their initial opinion: b_i = s_i.

sink_mask = ~non_sink
b_sink_pres = (1.0 - alpha_ref) * s.copy()
b_sink_pres[sink_mask] = s[sink_mask]  # sink nodes keep 100% of s_i

x_sink_pres, hist_sp = jacobi(A, b_sink_pres, tol=1e-8, criterion="residual", mode="matrix")

print(f"Standard model:        Jacobi iters = 114, std(x*) = {np.std(x_ref):.4f}, "
      f"std(x* on non-sinks) = {np.std(x_ref[non_sink]):.4f}, std(x* on sinks) = {np.std(x_ref[sink_mask]):.4f}")
print(f"Sink-preserving model: Jacobi iters = {len(hist_sp)}, std(x*) = {np.std(x_sink_pres):.4f}, "
      f"std(x* on non-sinks) = {np.std(x_sink_pres[non_sink]):.4f}, std(x* on sinks) = {np.std(x_sink_pres[sink_mask]):.4f}")

In [ ]:
# 3.3 Experiment: Radical dogmatic influencers (top 1% hubs with highest in-degree)
# In-degree = number of users who trust node j (column sum of adjacency pattern)

in_degrees = np.asarray((W > 0).sum(axis=0)).ravel()
n_hubs = max(1, int(0.01 * n))  # top 1% = 758 influencers
hub_indices = np.argsort(in_degrees)[-n_hubs:]

# Scenario A: Top 1% influencers are dogmatic extremists (s_i = +1, alpha_i = 0.01)
s_inf = s.copy()
s_inf[hub_indices] = 1.0
alpha_vec_inf = np.full(n, alpha_ref)
alpha_vec_inf[hub_indices] = 0.01  # dogmatic: 99% attached to their own +1 belief

A_inf = sp.eye(n, format="csr") - sp.diags(alpha_vec_inf) @ W
b_inf = (1.0 - alpha_vec_inf) * s_inf
x_inf, hist_inf = gauss_seidel(A_inf, b_inf, tol=1e-8, criterion="residual", mode="matrix")

# Scenario B: Control group - 1% random ordinary users become dogmatic extremists (s_i = +1, alpha_i = 0.01)
rng = np.random.default_rng(42)
random_indices = rng.choice(n, size=n_hubs, replace=False)
s_rnd = s.copy()
s_rnd[random_indices] = 1.0
alpha_vec_rnd = np.full(n, alpha_ref)
alpha_vec_rnd[random_indices] = 0.01

A_rnd = sp.eye(n, format="csr") - sp.diags(alpha_vec_rnd) @ W
b_rnd = (1.0 - alpha_vec_rnd) * s_rnd
x_rnd, hist_rnd = gauss_seidel(A_rnd, b_rnd, tol=1e-8, criterion="residual", mode="matrix")

print(f"Top 1% hubs average in-degree: {in_degrees[hub_indices].mean():.1f} followers (overall average: {in_degrees.mean():.1f})")
print(f"Baseline mean opinion:                     {np.mean(x_ref):+.4f}  (GS iters: 68)")
print(f"1% random dogmatic extremists (+1):        {np.mean(x_rnd):+.4f}  (GS iters: {len(hist_rnd)})")
print(f"1% hub dogmatic influencers (+1):          {np.mean(x_inf):+.4f}  (GS iters: {len(hist_inf)}) -> "
      f"{np.mean(x_inf) / max(1e-9, np.mean(x_rnd)):.1f}x stronger societal shift!")

In [ ]:
# 3.4 Visualizations of sociological experiments

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

# Panel 1: Opinion distributions across alpha
axes[0].hist(s, bins=40, density=True, alpha=0.25, color=ink, label="initial s (Uniform[-1,1])")
for a, col in [(0.30, aqua), (0.70, yellow), (0.85, blue), (0.95, orange)]:
    axes[0].hist(solutions_by_alpha[a], bins=45, density=True, histtype="step", lw=2, color=col, label=rf"$x^*$ ($\alpha={a}$)")
axes[0].set_xlabel("opinion value")
axes[0].set_ylabel("density")
axes[0].set_title("Opinion contraction vs susceptibility", loc="left")
axes[0].legend(frameon=False, fontsize=8)

# Panel 2: Echo-chamber scatter plot (user opinion vs trusted neighbors' average)
sample_idx = np.random.default_rng(42).choice(np.where(non_sink)[0], size=3000, replace=False)
axes[1].scatter(x_ref[sample_idx], (W @ x_ref)[sample_idx], s=6, alpha=0.3, color=blue)
lims = [-0.6, 0.6]
axes[1].plot(lims, lims, color=ink, ls="--", lw=1.2, label=r"$x_i^* = (Wx^*)_i$")
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_xlabel(r"user equilibrium opinion $x_i^*$")
axes[1].set_ylabel(r"trusted neighbors' average $(Wx^*)_i$")
axes[1].set_title(f"Echo-chamber alignment (r = {corr_equil:.2f})", loc="left")
axes[1].legend(frameon=False, fontsize=8.5)

# Panel 3: Radical influencer shift on the population opinion distribution
axes[2].hist(x_ref, bins=45, density=True, histtype="step", lw=2, color=ink, label=f"baseline (mean {np.mean(x_ref):+.3f})")
axes[2].hist(x_rnd, bins=45, density=True, histtype="step", lw=2, color=aqua, label=f"1% random zealots (mean {np.mean(x_rnd):+.3f})")
axes[2].hist(x_inf, bins=45, density=True, histtype="step", lw=2, color=orange, label=f"1% hub influencers (mean {np.mean(x_inf):+.3f})")
axes[2].set_xlabel("equilibrium opinion $x_i^*$")
axes[2].set_ylabel("density")
axes[2].set_title("Impact of dogmatic influencers (+1)", loc="left")
axes[2].legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(reports_folder / "m3_sociological_experiments.png", dpi=200)
plt.show()

### Results of Section 3

Figure: `reports/m3_sociological_experiments.png`.

1. **Depolarization vs Echo Chambers:** As social susceptibility $\alpha$ increases from $0.10$ to $0.98$, the standard deviation of opinions in the network drops from $0.525$ to $0.068$ (Panel 1). Because initial opinions $s_i \sim \mathcal{U}[-1, 1]$ are assigned independently of the graph topology, social averaging pulls users toward the global mean ($0$). However, local disagreement across trust links drops sharply, and the correlation between a user's equilibrium opinion $x_i^*$ and their trusted circle's average $(Wx^*)_i$ jumps from $r = 0.00$ initially to **$r = 0.65$** at equilibrium (Panel 2). Users become strongly aligned with their local trust neighborhood—the mathematical signature of **local echo chambers**.
2. **Standard vs Sink-Preserving Formulation:** When the $15{,}538$ sink nodes retain $100\%$ of their initial opinion ($b_i = s_i$ instead of $(1-\alpha)s_i$), the iteration matrix $B_J = \alpha W$ is unchanged, so Jacobi still converges in **almost the exact same number of iterations** ($120$ vs $114$). However, because sink nodes act as stubbon anchors Trusted by others, the overall variance of opinions in the network remains much higher ($\mathrm{std}(x^*) = 0.369$ vs $0.118$ in the standard model, and $0.258$ vs $0.123$ on non-sink users).
3. **Asymmetric Power of Dogmatic Influencers (Panel 3):** When $1\%$ of randomly chosen users become dogmatic extremists ($s_i = +1, \alpha_i = 0.01$), the average societal opinion shifts from $+0.0005$ to $+0.018$. When the **top $1\%$ most trusted hubs** adopt the exact same dogmatic stance, the mean societal opinion shifts to **$+0.252$**—over **$14$ times more impact**! Moreover, replacing $\alpha = 0.85$ with $\alpha_i = 0.01$ on the hubs strengthens diagonal dominance on the most connected rows of $A$, which actually **accelerates Gauss-Seidel convergence** from $68$ iterations down to $51$ iterations.